## ⚠️ Note on possible lookahead bias

This notebook trains on `NG_daily10.csv`, built from public sources (AGSI storage, ALSI LNG sendout, ENTSOG pipeline flows, energy-charts.info power data) that are sometimes **revised after initial publication**. The values used here are whatever is currently available from each API, not necessarily what would genuinely have been known in real time on each historical date. This is a common and hard-to-fully-eliminate source of lookahead bias in backtests built from revised public data, and it has **not** been corrected for here — treat backtested performance as an upper bound on what a live, real-time version of this model could achieve, not a faithful simulation of it.

`LNG_sendout(GWh/d)` gaps are filled via forward-only interpolation (`limit_direction='forward'`) upstream in `build_ng_daily_dataset.py`, which avoids filling a gap using a later value — but no other missing-data handling in the pipeline has been specifically audited for this.

The gradient boosting section below has been rebuilt to purge overlapping-horizon training labels (per López de Prado's *Advances in Financial Machine Learning*), use a genuine train/validation/test split rather than validating on the test set, and compound non-overlapping 7-day positions rather than daily-sampled overlapping ones. The Ridge and Ensemble sections further down have **not** been updated with the same fixes yet and still use the earlier (flawed) rolling-window methodology.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

df = pd.read_csv('NG_daily10.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df[df['Date'] >= '2022-01-01']
print(df.columns)

Index(['Date', 'JKM(USD/mmbtu)', 'HH(USD/mmbtu)', 'TTF(USD/mmbtu)',
       'Storage(TWh)', 'LNG_sendout(GWh/d)', 'corridor_BY', 'corridor_DE',
       'corridor_DZ', 'corridor_LY', 'corridor_MA', 'corridor_NO',
       'corridor_RU-Baltic', 'corridor_TR', 'corridor_UA', 'Berlin_HDD',
       'London_HDD', 'Rome_HDD', 'Berlin_CDD', 'London_CDD', 'Rome_CDD',
       'USD-EUR', 'VIX', 'Biomass', 'Cross border electricity trading',
       'Fossil brown coal / lignite', 'Fossil coal-derived gas', 'Fossil gas',
       'Fossil hard coal', 'Fossil oil', 'Fossil oil shale', 'Fossil peat',
       'Geothermal', 'Hydro Run-of-River', 'Hydro pumped storage',
       'Hydro pumped storage consumption', 'Hydro water reservoir', 'Load',
       'Nuclear', 'Other renewables', 'Others',
       'Renewable share of generation', 'Renewable share of load',
       'Residual load', 'Solar', 'Waste', 'Wind offshore', 'Wind onshore',
       'DE_price(EUR/MWh)'],
      dtype='str')


In [2]:
# Make new (composite) variables
df['HDD'] = df['Berlin_HDD'] + df['Rome_HDD'] + df['London_HDD']
df['Storage7'] = df['Storage(TWh)'].diff(7)
df['Storage30'] = df['Storage(TWh)'].diff(30)
# Sum whichever corridor_* columns are actually present -- corridor_RU-DE
# (Nordstream) in particular isn't always in the pipeline's output (its flows
# have been ~0 since 2022, so it's sometimes dropped upstream entirely), so
# this is written defensively rather than assuming a fixed column list.
corridor_cols = [c for c in df.columns if c.startswith('corridor_')]
print(f"Corridor columns found in this file: {corridor_cols}")
df['corridors'] = df[corridor_cols].sum(axis=1)

Corridor columns found in this file: ['corridor_BY', 'corridor_DE', 'corridor_DZ', 'corridor_LY', 'corridor_MA', 'corridor_NO', 'corridor_RU-Baltic', 'corridor_TR', 'corridor_UA']


In [ ]:
var = 'Date'
other_var = 'TTF(USD/mmbtu)'

fig = px.line(
    df, 
    x=var, 
    y=other_var,
    title= f'{other_var} v {var}',
)

fig.show()

In [ ]:
var = 'Date'
other_var = 'TTF(USD/mmbtu)'

fig = px.line(
    df, 
    x=var, 
    y=df[other_var].diff(),
    title= f'First differenced {other_var} v {var}',
)

fig.show()

In [ ]:
var = 'Date'
other_var = 'TTF(USD/mmbtu)'
df['log_' + other_var] = np.log(df[other_var]).diff()

fig = px.line(
    df, 
    x=var, 
    y=df['log_' + other_var],
    title= f'First difference log({other_var}) v {var}',
)

fig.show()

In [ ]:
df['log_TTF'] = np.log(df['TTF(USD/mmbtu)']).diff()
result = adfuller(df['log_TTF'].dropna())

print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print("Critical Values:")
for key, value in result[4].items():
  print(f"   {key}: {value:.4f}")

## XGBoost Gradient Boosting (purged, expanding-window validation, 5-row horizon)

In [3]:
import itertools

import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
HORIZON_DAYS = 5             # fixed at 5 rows, not searched -- matches the Ridge
                              # section below so the two can be combined into an ensemble
MIN_TRAIN_YEARS = 2
TRAIN_VAL_CAP = pd.Timestamp('2025-05-31')
TEST_START = pd.Timestamp('2025-06-01')
VALIDATION_FOLD_DAYS = 90

MAX_DEPTH_CANDIDATES = [2, 3, 4, 6, 8]
LEARNING_RATE_CANDIDATES = [0.01, 0.03, 0.05, 0.1]

XGB_FIXED_PARAMS = dict(
    random_state=42,
    objective='reg:absoluteerror',
    verbosity=0,
)

features = [
    'Storage(TWh)',
    'HDD',
    'corridors',
    'LNG_sendout(GWh/d)',
    'DE_price(EUR/MWh)',
    'momentum',        # NEW: realized HORIZON_DAYS-row return as of the current row
]

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# NOTE: NG_daily10.csv contains trading days only, so shift(-HORIZON_DAYS) moves
# HORIZON_DAYS ROWS ahead, not HORIZON_DAYS calendar days. 5 rows is close to one
# genuine calendar week (~5 trading days/week). target_end_date tracks the ACTUAL
# calendar date each row's target reaches, so purging is exact regardless of
# holidays nudging that gap around week to week.
df['target_end_date'] = df['Date'].shift(-HORIZON_DAYS)
df['dlog_TTF'] = np.log(df['TTF(USD/mmbtu)']).shift(-HORIZON_DAYS) - np.log(df['TTF(USD/mmbtu)'])

# momentum: the return that ALREADY HAPPENED over the prior HORIZON_DAYS rows,
# ending at the current row -- uses only past-and-current data, no leakage.
df['momentum'] = np.log(df['TTF(USD/mmbtu)']) - np.log(df['TTF(USD/mmbtu)']).shift(HORIZON_DAYS)

target = 'dlog_TTF'
data = df.dropna(subset=[target, 'target_end_date'] + features).reset_index(drop=True)

train_val_pool = data[
    (data['target_end_date'] < TEST_START) & (data['Date'] <= TRAIN_VAL_CAP)
].reset_index(drop=True)
test_pool = data[data['Date'] >= TEST_START].reset_index(drop=True)

data_start = train_val_pool['Date'].min()
min_train_end = data_start + pd.DateOffset(years=MIN_TRAIN_YEARS)

# --------------------------------------------------------------------------- #
# max_depth x learning_rate search through purged expanding-window folds.
# n_estimators resolved by early stopping within each combination. Ranked on
# validation R^2 vs. a "predict no change" naive benchmark.
# --------------------------------------------------------------------------- #
grid = list(itertools.product(MAX_DEPTH_CANDIDATES, LEARNING_RATE_CANDIDATES))
print(f"Searching {len(grid)} (max_depth, learning_rate) combinations at a fixed "
      f"{HORIZON_DAYS}-row horizon, across several expanding-window folds, "
      f"purged by actual target reach, with momentum included as a feature...\n")

fold_results = []
for max_depth, learning_rate in grid:
    fold_val_start = min_train_end
    fold_num = 0

    while fold_val_start < train_val_pool['Date'].max():
        fold_val_end = fold_val_start + pd.Timedelta(days=VALIDATION_FOLD_DAYS)

        fold_train_mask = train_val_pool['target_end_date'] < fold_val_start
        fold_val_mask = (train_val_pool['Date'] >= fold_val_start) & (train_val_pool['Date'] < fold_val_end)

        X_tr = train_val_pool.loc[fold_train_mask, features]
        y_tr = train_val_pool.loc[fold_train_mask, target]
        X_val = train_val_pool.loc[fold_val_mask, features]
        y_val = train_val_pool.loc[fold_val_mask, target]

        if len(X_tr) >= 100 and len(X_val) >= 10:
            fold_num += 1
            fold_model = xgb.XGBRegressor(
                n_estimators=500, max_depth=max_depth, learning_rate=learning_rate,
                early_stopping_rounds=50, eval_metric='rmse', **XGB_FIXED_PARAMS,
            )
            fold_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            val_pred = fold_model.predict(X_val)

            sse_model = np.sum((y_val.values - val_pred) ** 2)
            sse_naive = np.sum(y_val.values ** 2)
            val_r2_vs_naive = 1 - sse_model / sse_naive if sse_naive > 0 else np.nan

            fold_results.append(dict(
                max_depth=max_depth, learning_rate=learning_rate, fold=fold_num,
                best_iteration=fold_model.best_iteration, val_r2_vs_naive=val_r2_vs_naive,
            ))

        fold_val_start = fold_val_end

fold_results_df = pd.DataFrame(fold_results)
summary = fold_results_df.groupby(['max_depth', 'learning_rate']).agg(
    mean_val_r2=('val_r2_vs_naive', 'mean'),
    median_best_iteration=('best_iteration', 'median'),
    n_folds=('fold', 'count'),
).sort_values('mean_val_r2', ascending=False)
print("Validation results by (max_depth, learning_rate):")
print(summary.to_string())

best_combo = summary.index[0]
best_row = summary.iloc[0]
XGB_PARAMS = {**XGB_FIXED_PARAMS, 'max_depth': int(best_combo[0]), 'learning_rate': float(best_combo[1])}
final_n_estimators = int(best_row['median_best_iteration'])
print(f"\nChosen: max_depth={XGB_PARAMS['max_depth']}, learning_rate={XGB_PARAMS['learning_rate']}, "
      f"n_estimators={final_n_estimators} (highest mean validation R2 vs naive)")

# --------------------------------------------------------------------------- #
# Fit the final model on the full purged train+validation pool
# --------------------------------------------------------------------------- #
final_model = xgb.XGBRegressor(n_estimators=final_n_estimators, **XGB_PARAMS)
final_model.fit(train_val_pool[features], train_val_pool[target])
print(f"\nFinal model fit on {len(train_val_pool)} rows "
      f"({train_val_pool['Date'].min().date()} to {train_val_pool['Date'].max().date()})")

importance_df = pd.DataFrame(
    {'Feature': features, 'Importance': final_model.feature_importances_}
).sort_values(by='Importance', ascending=False)
print('\nFeature Importance:')
print(importance_df.to_string(index=False))

# --------------------------------------------------------------------------- #
# Test: a NEW position every HORIZON_DAYS rows, non-overlapping, never seen
# by the model above in any way
# --------------------------------------------------------------------------- #
decision_positions = list(range(0, len(test_pool), HORIZON_DAYS))
test_sample = test_pool.iloc[decision_positions].reset_index(drop=True)

X_test = test_sample[features]
y_test = test_sample[target]
predictions = final_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
corr = np.corrcoef(y_test, predictions)[0, 1]
hit_rate = np.mean(np.sign(y_test) == np.sign(predictions)) * 100

print(f"\n--- Test Evaluation ({TEST_START.date()} onward, non-overlapping {HORIZON_DAYS}-row decisions) ---")
print(f"Number of independent test decisions: {len(test_sample)}"
      f"  (small sample -- treat these numbers as indicative, not precise)")
print(f"Test RMSE: {rmse:.5f}")
print(f"Correlation: {corr:.4f}")
print(f"Directional Hit Rate: {hit_rate:.2f}%")
print(f"Actual target std: {y_test.std():.5f}")
print(f"Prediction std: {predictions.std():.5f}")

# --------------------------------------------------------------------------- #
# Backtest with genuinely non-overlapping compounding. Sharpe annualization
# uses the actual mean calendar span per decision, not an assumed value.
# --------------------------------------------------------------------------- #
backtest_df = pd.DataFrame({
    'Date': test_sample['Date'].values,
    'Actual_Return': y_test.values,
    'Predicted_Return': predictions,
})
backtest_df['Position'] = np.where(backtest_df['Predicted_Return'] > 0, 1, 0)
backtest_df['Strategy_Return'] = backtest_df['Position'] * backtest_df['Actual_Return']

backtest_df['Cumulative_Market'] = np.exp(backtest_df['Actual_Return'].cumsum()) - 1
backtest_df['Cumulative_Strategy'] = np.exp(backtest_df['Strategy_Return'].cumsum()) - 1

total_strategy_return = backtest_df['Cumulative_Strategy'].iloc[-1]
total_market_return = backtest_df['Cumulative_Market'].iloc[-1]

avg_calendar_days_per_decision = (test_sample['Date'].iloc[-1] - test_sample['Date'].iloc[0]).days / max(len(test_sample) - 1, 1)
periods_per_year = 365 / avg_calendar_days_per_decision
strategy_std = backtest_df['Strategy_Return'].std()
sharpe_ratio = (backtest_df['Strategy_Return'].mean() / strategy_std) * np.sqrt(periods_per_year) if strategy_std > 0 else 0

print('\n--- Backtest Results (Non-overlapping Long/Cash Strategy) ---')
print(f'Total Strategy Return: {total_strategy_return * 100:.2f}%')
print(f'Total Buy-and-Hold Return: {total_market_return * 100:.2f}%')
print(f'Annualized Sharpe Ratio: {sharpe_ratio:.2f}  '
      f'(annualized using {periods_per_year:.1f} periods/year, '
      f'based on {avg_calendar_days_per_decision:.1f} actual calendar days/decision)')


Searching 20 (max_depth, learning_rate) combinations at a fixed 5-row horizon, across several expanding-window folds, purged by actual target reach, with momentum included as a feature...

Validation results by (max_depth, learning_rate):
                         mean_val_r2  median_best_iteration  n_folds
max_depth learning_rate                                             
2         0.10             -0.016508                   22.5        6
          0.05             -0.028994                    5.0        6
          0.03             -0.030931                   11.5        6
          0.01             -0.033222                   35.0        6
3         0.01             -0.049037                   18.5        6
          0.05             -0.050191                   23.0        6
6         0.03             -0.053934                   10.5        6
8         0.01             -0.054194                   81.0        6
3         0.03             -0.059192                   12.0        6
6 

## Ridge Regression (purged, expanding-window validation, 5-row horizon)

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
HORIZON_DAYS = 5             # fixed at 5 rows, not searched -- ~1 calendar week given
                              # trading-day-only data (see note below), matches the
                              # XGBoost section above so the two can be combined into an ensemble
MIN_TRAIN_YEARS = 2
TRAIN_VAL_CAP = pd.Timestamp('2025-05-31')
TEST_START = pd.Timestamp('2025-06-01')
VALIDATION_FOLD_DAYS = 90
ALPHA_CANDIDATES = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

features = [
    'Storage(TWh)',   # raw level, not Storage7/Storage30 differences
    'HDD',
    'corridors',
    'LNG_sendout(GWh/d)',
    'DE_price(EUR/MWh)',
    'momentum',        # NEW: realized HORIZON_DAYS-row return as of the current row
]

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# NOTE: NG_daily10.csv contains trading days only, so shift(-HORIZON_DAYS) moves
# HORIZON_DAYS ROWS ahead, not HORIZON_DAYS calendar days. 5 rows is close to one
# genuine calendar week (~5 trading days/week). target_end_date tracks the ACTUAL
# calendar date each row's target reaches, so purging is exact regardless of
# holidays nudging that gap around week to week.
df['target_end_date'] = df['Date'].shift(-HORIZON_DAYS)
df['dlog_TTF'] = np.log(df['TTF(USD/mmbtu)']).shift(-HORIZON_DAYS) - np.log(df['TTF(USD/mmbtu)'])

# momentum: the return that ALREADY HAPPENED over the prior HORIZON_DAYS rows,
# ending at the current row -- uses only past-and-current data, no leakage.
# Not deseasonalized, unlike the monthly model's momentum term, to keep it at
# the same (raw) treatment level as every other feature in this daily model.
df['momentum'] = np.log(df['TTF(USD/mmbtu)']) - np.log(df['TTF(USD/mmbtu)']).shift(HORIZON_DAYS)

target = 'dlog_TTF'
data = df.dropna(subset=[target, 'target_end_date'] + features).reset_index(drop=True)

# purge: a row belongs in train+val only if its target's ACTUAL end-date is
# before TEST_START (not an assumed calendar-day offset)
train_val_pool = data[
    (data['target_end_date'] < TEST_START) & (data['Date'] <= TRAIN_VAL_CAP)
].reset_index(drop=True)
test_pool = data[data['Date'] >= TEST_START].reset_index(drop=True)

data_start = train_val_pool['Date'].min()
min_train_end = data_start + pd.DateOffset(years=MIN_TRAIN_YEARS)

# --------------------------------------------------------------------------- #
# Alpha search through purged expanding-window folds. Same target_end_date
# logic is used for the fold-internal purge, not a calendar-day Timedelta.
# --------------------------------------------------------------------------- #
fold_results = []

print(f"Running expanding-window validation folds over alpha in {ALPHA_CANDIDATES} "
      f"(min {MIN_TRAIN_YEARS}yr train, {VALIDATION_FOLD_DAYS}-day folds, "
      f"purged by actual target reach), with momentum included as a feature...")

for alpha in ALPHA_CANDIDATES:
    fold_val_start = min_train_end
    fold_num = 0

    while fold_val_start < train_val_pool['Date'].max():
        fold_val_end = fold_val_start + pd.Timedelta(days=VALIDATION_FOLD_DAYS)

        fold_train_mask = train_val_pool['target_end_date'] < fold_val_start
        fold_val_mask = (train_val_pool['Date'] >= fold_val_start) & (train_val_pool['Date'] < fold_val_end)

        X_tr = train_val_pool.loc[fold_train_mask, features]
        y_tr = train_val_pool.loc[fold_train_mask, target]
        X_val = train_val_pool.loc[fold_val_mask, features]
        y_val = train_val_pool.loc[fold_val_mask, target]

        if len(X_tr) >= 100 and len(X_val) >= 10:
            fold_num += 1
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_val_scaled = scaler.transform(X_val)

            fold_model = Ridge(alpha=alpha, random_state=42)
            fold_model.fit(X_tr_scaled, y_tr)
            val_pred = fold_model.predict(X_val_scaled)

            sse_model = np.sum((y_val.values - val_pred) ** 2)
            sse_naive = np.sum(y_val.values ** 2)
            val_r2_vs_naive = 1 - sse_model / sse_naive if sse_naive > 0 else np.nan

            fold_results.append(dict(alpha=alpha, fold=fold_num, val_r2_vs_naive=val_r2_vs_naive))

        fold_val_start = fold_val_end

fold_results_df = pd.DataFrame(fold_results)
summary = fold_results_df.groupby('alpha').agg(
    mean_val_r2=('val_r2_vs_naive', 'mean'),
    n_folds=('fold', 'count'),
).sort_values('mean_val_r2', ascending=False)
print(f"\nValidation results by alpha:")
print(summary.to_string())

RIDGE_ALPHA = float(summary.index[0])
print(f"\nChosen alpha={RIDGE_ALPHA} (highest mean validation R2 vs naive)")

# --------------------------------------------------------------------------- #
# Fit the final model on the full purged train+validation pool
# --------------------------------------------------------------------------- #
final_scaler = StandardScaler()
X_train_val_scaled = final_scaler.fit_transform(train_val_pool[features])
final_model = Ridge(alpha=RIDGE_ALPHA, random_state=42)
final_model.fit(X_train_val_scaled, train_val_pool[target])
print(f"\nFinal model fit on {len(train_val_pool)} rows "
      f"({train_val_pool['Date'].min().date()} to {train_val_pool['Date'].max().date()})")

coef_df = pd.DataFrame(
    {'Feature': features, 'Coefficient': final_model.coef_}
).sort_values(by='Coefficient', key=np.abs, ascending=False)
print('\nStandardized Coefficients (Ridge, on scaled features -- comparable to each other by magnitude):')
print(coef_df.to_string(index=False))

# --------------------------------------------------------------------------- #
# Test: a NEW position every HORIZON_DAYS rows (non-overlapping), never seen
# by the model or scaler above in any way
# --------------------------------------------------------------------------- #
decision_positions = list(range(0, len(test_pool), HORIZON_DAYS))
test_sample = test_pool.iloc[decision_positions].reset_index(drop=True)

X_test_scaled = final_scaler.transform(test_sample[features])
y_test = test_sample[target]
predictions = final_model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
corr = np.corrcoef(y_test, predictions)[0, 1]
hit_rate = np.mean(np.sign(y_test) == np.sign(predictions)) * 100

print(f"\n--- Test Evaluation ({TEST_START.date()} onward, non-overlapping {HORIZON_DAYS}-row decisions) ---")
print(f"Number of independent test decisions: {len(test_sample)}"
      f"  (small sample -- treat these numbers as indicative, not precise)")
print(f"Test RMSE: {rmse:.5f}")
print(f"Correlation: {corr:.4f}")
print(f"Directional Hit Rate: {hit_rate:.2f}%")
print(f"Actual target std: {y_test.std():.5f}")
print(f"Prediction std: {predictions.std():.5f}")

# --------------------------------------------------------------------------- #
# Backtest with genuinely non-overlapping compounding
# --------------------------------------------------------------------------- #
backtest_df = pd.DataFrame({
    'Date': test_sample['Date'].values,
    'Actual_Return': y_test.values,
    'Predicted_Return': predictions,
})
backtest_df['Position'] = np.where(backtest_df['Predicted_Return'] > 0, 1, 0)
backtest_df['Strategy_Return'] = backtest_df['Position'] * backtest_df['Actual_Return']

backtest_df['Cumulative_Market'] = np.exp(backtest_df['Actual_Return'].cumsum()) - 1
backtest_df['Cumulative_Strategy'] = np.exp(backtest_df['Strategy_Return'].cumsum()) - 1

total_strategy_return = backtest_df['Cumulative_Strategy'].iloc[-1]
total_market_return = backtest_df['Cumulative_Market'].iloc[-1]

avg_calendar_days_per_decision = (test_sample['Date'].iloc[-1] - test_sample['Date'].iloc[0]).days / max(len(test_sample) - 1, 1)
periods_per_year = 365 / avg_calendar_days_per_decision
strategy_std = backtest_df['Strategy_Return'].std()
sharpe_ratio = (backtest_df['Strategy_Return'].mean() / strategy_std) * np.sqrt(periods_per_year) if strategy_std > 0 else 0

print('\n--- Backtest Results (Non-overlapping Long/Cash Strategy) ---')
print(f'Total Strategy Return: {total_strategy_return * 100:.2f}%')
print(f'Total Buy-and-Hold Return: {total_market_return * 100:.2f}%')
print(f'Annualized Sharpe Ratio: {sharpe_ratio:.2f}  '
      f'(annualized using {periods_per_year:.1f} periods/year, '
      f'based on {avg_calendar_days_per_decision:.1f} actual calendar days/decision)')


Running expanding-window validation folds over alpha in [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0] (min 2yr train, 90-day folds, purged by actual target reach), with momentum included as a feature...

Validation results by alpha:
         mean_val_r2  n_folds
alpha                        
1000.00     0.032660        6
100.00     -0.073685        6
10.00      -0.153747        6
1.00       -0.165118        6
0.10       -0.166300        6
0.01       -0.166419        6

Chosen alpha=1000.0 (highest mean validation R2 vs naive)

Final model fit on 844 rows (2022-01-10 to 2025-05-22)

Standardized Coefficients (Ridge, on scaled features -- comparable to each other by magnitude):
           Feature  Coefficient
               HDD    -0.011193
 DE_price(EUR/MWh)    -0.008876
LNG_sendout(GWh/d)    -0.005728
         corridors     0.004751
      Storage(TWh)    -0.002927
          momentum     0.000246

--- Test Evaluation (2025-06-01 onward, non-overlapping 5-row decisions) ---
Number of independent

## Ensemble (XGBoost + Ridge, validated blend weight, 5-row horizon)

In [5]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------- #
# This blends the XGBoost and Ridge models from the two sections above. It
# reuses each model's already-validated hyperparameters (XGB_PARAMS,
# final_n_estimators, RIDGE_ALPHA) rather than re-running those searches --
# run the XGBoost and Ridge cells above first, in order. The only new thing
# actually being tuned here is the blend WEIGHT between the two models'
# predictions, chosen through the same purged expanding-window folds used
# throughout this notebook, not guessed or fixed at an arbitrary 70/30 split.
# --------------------------------------------------------------------------- #
for _name in ['XGB_PARAMS', 'final_n_estimators', 'RIDGE_ALPHA']:
    if _name not in globals():
        raise NameError(f"{_name} not found -- run the XGBoost and Ridge cells above first.")

HORIZON_DAYS = 5
MIN_TRAIN_YEARS = 2
TRAIN_VAL_CAP = pd.Timestamp('2025-05-31')
TEST_START = pd.Timestamp('2025-06-01')
VALIDATION_FOLD_DAYS = 90
BLEND_WEIGHT_CANDIDATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # weight on XGBoost; (1 - w) on Ridge

features = ['Storage(TWh)', 'HDD', 'corridors', 'LNG_sendout(GWh/d)', 'DE_price(EUR/MWh)', 'momentum']

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df['target_end_date'] = df['Date'].shift(-HORIZON_DAYS)
df['dlog_TTF'] = np.log(df['TTF(USD/mmbtu)']).shift(-HORIZON_DAYS) - np.log(df['TTF(USD/mmbtu)'])

# momentum: the return that ALREADY HAPPENED over the prior HORIZON_DAYS rows,
# ending at the current row -- uses only past-and-current data, no leakage.
df['momentum'] = np.log(df['TTF(USD/mmbtu)']) - np.log(df['TTF(USD/mmbtu)']).shift(HORIZON_DAYS)

target = 'dlog_TTF'
data = df.dropna(subset=[target, 'target_end_date'] + features).reset_index(drop=True)

train_val_pool = data[
    (data['target_end_date'] < TEST_START) & (data['Date'] <= TRAIN_VAL_CAP)
].reset_index(drop=True)
test_pool = data[data['Date'] >= TEST_START].reset_index(drop=True)

data_start = train_val_pool['Date'].min()
min_train_end = data_start + pd.DateOffset(years=MIN_TRAIN_YEARS)

# --------------------------------------------------------------------------- #
# Blend weight search through purged expanding-window folds. Both models use
# their already-chosen hyperparameters here (no early stopping needed for
# XGBoost since n_estimators is already fixed) -- only the weight is searched.
# --------------------------------------------------------------------------- #
fold_results = []
fold_val_start = min_train_end
fold_num = 0

print("Searching blend weight (0 = all Ridge, 1 = all XGBoost) through purged expanding-window folds...")

while fold_val_start < train_val_pool['Date'].max():
    fold_val_end = fold_val_start + pd.Timedelta(days=VALIDATION_FOLD_DAYS)

    fold_train_mask = train_val_pool['target_end_date'] < fold_val_start
    fold_val_mask = (train_val_pool['Date'] >= fold_val_start) & (train_val_pool['Date'] < fold_val_end)

    X_tr = train_val_pool.loc[fold_train_mask, features]
    y_tr = train_val_pool.loc[fold_train_mask, target]
    X_val = train_val_pool.loc[fold_val_mask, features]
    y_val = train_val_pool.loc[fold_val_mask, target]

    if len(X_tr) >= 100 and len(X_val) >= 10:
        fold_num += 1

        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        ridge_model = Ridge(alpha=RIDGE_ALPHA, random_state=42)
        ridge_model.fit(X_tr_scaled, y_tr)
        ridge_pred = ridge_model.predict(X_val_scaled)

        xgb_model = xgb.XGBRegressor(n_estimators=final_n_estimators, **XGB_PARAMS)
        xgb_model.fit(X_tr, y_tr)
        xgb_pred = xgb_model.predict(X_val)

        for w in BLEND_WEIGHT_CANDIDATES:
            blended_pred = w * xgb_pred + (1 - w) * ridge_pred
            sse_model = np.sum((y_val.values - blended_pred) ** 2)
            sse_naive = np.sum(y_val.values ** 2)
            val_r2_vs_naive = 1 - sse_model / sse_naive if sse_naive > 0 else np.nan
            fold_results.append(dict(weight=w, fold=fold_num, val_r2_vs_naive=val_r2_vs_naive))

    fold_val_start = fold_val_end

fold_results_df = pd.DataFrame(fold_results)
summary = fold_results_df.groupby('weight').agg(
    mean_val_r2=('val_r2_vs_naive', 'mean'),
    n_folds=('fold', 'count'),
).sort_values('mean_val_r2', ascending=False)
print(summary.to_string())

BLEND_WEIGHT = float(summary.index[0])
print(f"\nChosen blend weight: {BLEND_WEIGHT:.1f} on XGBoost, {1 - BLEND_WEIGHT:.1f} on Ridge "
      f"(highest mean validation R2 vs naive)")
if BLEND_WEIGHT in (0.0, 1.0):
    print("Note: the search picked an ENDPOINT weight -- i.e. it found no validation benefit "
          "to blending at all, just 'use whichever single model was better.' That's a valid "
          "and informative result, not a failure of the search.")

# --------------------------------------------------------------------------- #
# Fit both final models on the full purged train+validation pool
# --------------------------------------------------------------------------- #
final_scaler = StandardScaler()
X_train_val_scaled = final_scaler.fit_transform(train_val_pool[features])
final_ridge_model = Ridge(alpha=RIDGE_ALPHA, random_state=42)
final_ridge_model.fit(X_train_val_scaled, train_val_pool[target])

final_xgb_model = xgb.XGBRegressor(n_estimators=final_n_estimators, **XGB_PARAMS)
final_xgb_model.fit(train_val_pool[features], train_val_pool[target])

print(f"\nBoth final models fit on {len(train_val_pool)} rows "
      f"({train_val_pool['Date'].min().date()} to {train_val_pool['Date'].max().date()})")

# --------------------------------------------------------------------------- #
# Test: a NEW position every HORIZON_DAYS rows, non-overlapping, never seen
# by either model in any way
# --------------------------------------------------------------------------- #
decision_positions = list(range(0, len(test_pool), HORIZON_DAYS))
test_sample = test_pool.iloc[decision_positions].reset_index(drop=True)

X_test = test_sample[features]
X_test_scaled = final_scaler.transform(X_test)
y_test = test_sample[target]

ridge_pred_test = final_ridge_model.predict(X_test_scaled)
xgb_pred_test = final_xgb_model.predict(X_test)
predictions = BLEND_WEIGHT * xgb_pred_test + (1 - BLEND_WEIGHT) * ridge_pred_test

print(f"\n--- Test Evaluation ({TEST_START.date()} onward, non-overlapping {HORIZON_DAYS}-row decisions) ---")
print(f"Number of independent test decisions: {len(test_sample)}"
      f"  (small sample -- treat these numbers as indicative, not precise)")

for name, pred in [('Ridge alone', ridge_pred_test), ('XGBoost alone', xgb_pred_test), ('Ensemble blend', predictions)]:
    rmse_i = np.sqrt(mean_squared_error(y_test, pred))
    corr_i = np.corrcoef(y_test, pred)[0, 1]
    hit_i = np.mean(np.sign(y_test) == np.sign(pred)) * 100
    print(f"  {name:15s}: RMSE={rmse_i:.5f}  corr={corr_i:+.4f}  hit_rate={hit_i:.2f}%  pred_std={pred.std():.5f}")

print(f"\nActual target std: {y_test.std():.5f}")

# --------------------------------------------------------------------------- #
# Backtest with genuinely non-overlapping compounding, using the ensemble
# blend's predictions
# --------------------------------------------------------------------------- #
backtest_df = pd.DataFrame({
    'Date': test_sample['Date'].values,
    'Actual_Return': y_test.values,
    'Predicted_Return': predictions,
})
backtest_df['Position'] = np.where(backtest_df['Predicted_Return'] > 0, 1, 0)
backtest_df['Strategy_Return'] = backtest_df['Position'] * backtest_df['Actual_Return']

backtest_df['Cumulative_Market'] = np.exp(backtest_df['Actual_Return'].cumsum()) - 1
backtest_df['Cumulative_Strategy'] = np.exp(backtest_df['Strategy_Return'].cumsum()) - 1

total_strategy_return = backtest_df['Cumulative_Strategy'].iloc[-1]
total_market_return = backtest_df['Cumulative_Market'].iloc[-1]

avg_calendar_days_per_decision = (test_sample['Date'].iloc[-1] - test_sample['Date'].iloc[0]).days / max(len(test_sample) - 1, 1)
periods_per_year = 365 / avg_calendar_days_per_decision
strategy_std = backtest_df['Strategy_Return'].std()
sharpe_ratio = (backtest_df['Strategy_Return'].mean() / strategy_std) * np.sqrt(periods_per_year) if strategy_std > 0 else 0

print('\n--- Backtest Results (Non-overlapping Long/Cash Strategy, Ensemble Blend) ---')
print(f'Total Strategy Return: {total_strategy_return * 100:.2f}%')
print(f'Total Buy-and-Hold Return: {total_market_return * 100:.2f}%')
print(f'Annualized Sharpe Ratio: {sharpe_ratio:.2f}  '
      f'(annualized using {periods_per_year:.1f} periods/year, '
      f'based on {avg_calendar_days_per_decision:.1f} actual calendar days/decision)')


Searching blend weight (0 = all Ridge, 1 = all XGBoost) through purged expanding-window folds...
        mean_val_r2  n_folds
weight                      
0.1        0.032973        6
0.0        0.032660        6
0.2        0.030493        6
0.3        0.025221        6
0.4        0.017157        6
0.5        0.006300        6
0.6       -0.007349        6
0.7       -0.023790        6
0.8       -0.043024        6
0.9       -0.065051        6
1.0       -0.089870        6

Chosen blend weight: 0.1 on XGBoost, 0.9 on Ridge (highest mean validation R2 vs naive)

Both final models fit on 844 rows (2022-01-10 to 2025-05-22)

--- Test Evaluation (2025-06-01 onward, non-overlapping 5-row decisions) ---
Number of independent test decisions: 57  (small sample -- treat these numbers as indicative, not precise)
  Ridge alone    : RMSE=0.09765  corr=-0.0256  hit_rate=50.88%  pred_std=0.01577
  XGBoost alone  : RMSE=0.09756  corr=+0.0694  hit_rate=54.39%  pred_std=0.02676
  Ensemble blend : RMSE=0.09